In [1]:
import requests
import random

from fake_useragent import UserAgent

import scrapy
from scrapy.crawler import CrawlerRunner
import re

from crochet import setup, wait_for
setup()

In [2]:
import scrapy
import logging

def extractText(regex, text, G=0):
    print(text)
    try:
        return re.search(regex, text).group(G)
    except:
        return ''

class QuotesToCsv(scrapy.Spider):
    name = "WebScraper"
    start_urls = [
        'https://myanimelist.net/people/118'
    ]

    proxies = [
        'http://proxy1.com:port',
    ]
    
    custom_settings = {
        'FEEDS': {
            'OUTPUT_FILENAME.csv': {
                'format': 'csv',
                'overwrite': True
            }
        },
    }

    '''
    def start_requests(self):
        for url in self.start_urls:
            yield scrapy.Request(url,
                                 meta= {'proxy':random.choice(self.proxies)}
                                 )
    '''
    
    def parse(self, response):
        print(f"Parsing started for: {response.url}")  # Custom print message
        
        for tr in response.css('tr.js-people-character'):
            td1 = tr.css('td:nth-child(2)')
            info_text = td1.css('div.spaceit_pad.anime-info-text::text').get()

            td2 = tr.css('td:nth-child(3)')

            
            yield {
                'Title': td1.css('a.js-people-title::text').get().strip(),
                'Link': td1.css('a.js-people-title::attr(href)').get(),  
                'ID' : int(re.search(r'/anime/(\d+)', td1.css('a.js-people-title::attr(href)').get()).group(1)),
                'Year' :  extractText(regex=r'(\d+)$', text=info_text),
                'Season' : extractText(regex=r'(?<=,)(.*?)(?=\s\d)', text=info_text).strip(),
                'Type' : extractText(regex=r'^[^,]+', text=info_text),

                'Character' : td2.css('a::text').get().strip(),
                'CharacterLink' : td2.css('a::attr(href)').get(),
                'CharacterID' : int(re.search(r'/character/(\d+)', td2.css('a::attr(href)').get()).group(1)),
                'CharacterRole' : extractText(regex=r'(?<=^)\S+|\S+(?=$)', text=td2.css('div.spaceit_pad:nth-child(2)::text').get()).strip(),
                'CharacterFavorites' : int(td2.css('div.spaceit_pad.character-total-favorites::text').get().split()[0].replace(',',''))
                
            }

        print(f"Parsing finished for: {response.url}")  # Custom print message


In [3]:
@wait_for(10)
def run_spider():
    crawler = CrawlerRunner()
    d = crawler.crawl(QuotesToCsv)
    return d

run_spider()